# Multilingual Health QA — End-to-End NLLB-200 & Hybrid RAG Pipeline

**Challenge:** Zindi Multilingual Health Question Answering in Low-Resource African Languages

**Repository:** [SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages](https://github.com/SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages.git)

### Complete Goals & Features Implemented:
1. **Generation Model (`facebook/nllb-200-distilled-600M`)**: Native FLORES-200 language code tokens (`aka_Latn`, `amh_Ethi`, `lug_Latn`, `swh_Latn`, `eng_Latn`) with per-language mini-batch grouping.
2. **Sentence-Embedding & Hybrid RAG Retrieval**: Blends lexical TF-IDF subword n-grams and multilingual dense embeddings (`sentence-transformers/paraphrase-multilingual-mpnet-base-v2`) in `src/retrieval.py`.
3. **Tightened Threshold Optimization**: Fine-grained `0.01` grid search and Logistic Regression score calibrator in `src/threshold_optimizer.py`.
4. **Per-Language Epoch ROUGE Callback**: Live per-subset ROUGE-1 & ROUGE-L table breakdown during fine-tuning.
5. **Validation-First Workflow**: Evaluates validation metrics first. Test submission files are generated **ONLY** when validation results are promising!

In [ ]:
# 1. Environment & Repository Setup (Universal Home Root Reset)
import os, sys, shutil, zipfile, urllib.request
from pathlib import Path

# Reset working directory to home root to avoid stale CWD or nested directories
try:
    home_dir = Path.home()
    if (Path('/home/jovyan')).exists():
        home_dir = Path('/home/jovyan')
    elif (Path('/content')).exists():
        home_dir = Path('/content')
    os.chdir(home_dir)
except Exception as e:
    print(f'[WARN] Directory reset fallback: {e}')

repo_name = 'Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages'
zip_url = 'https://github.com/SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages/archive/refs/heads/main.zip'

# Clean up any previous folder to ensure fresh code and datasets
if os.path.exists(repo_name):
    print(f"Cleaning previous '{repo_name}' directory...")
    shutil.rmtree(repo_name, ignore_errors=True)

if os.path.exists('repo.zip'):
    os.remove('repo.zip')

print('Downloading repository & raw datasets from GitHub...')
urllib.request.urlretrieve(zip_url, 'repo.zip')

print('Extracting project files...')
with zipfile.ZipFile('repo.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

if os.path.exists(f'{repo_name}-main'):
    os.rename(f'{repo_name}-main', repo_name)

%cd {repo_name}
print('\nSetup complete! Project root:')
!pwd


In [ ]:
# 2. Install Required Dependencies
!pip install -q torch transformers datasets evaluate scikit-learn pandas numpy rouge-score sentence-transformers accelerate peft
print('All package dependencies installed successfully!')


In [ ]:
# Universal Self-Healing Path Setup (Prevents FileNotFoundError on deleted CWD)
import os, sys
from pathlib import Path

try:
    _cwd = Path.cwd()
except (FileNotFoundError, OSError):
    os.chdir('/home/jovyan')
    _cwd = Path.cwd()

repo_name = 'Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages'

if (_cwd / 'src').exists():
    BASE_DIR = _cwd
elif (_cwd.parent / 'src').exists():
    BASE_DIR = _cwd.parent
elif (_cwd / repo_name / 'src').exists():
    BASE_DIR = _cwd / repo_name
elif (Path('/home/jovyan') / repo_name / 'src').exists():
    BASE_DIR = Path('/home/jovyan') / repo_name
else:
    BASE_DIR = Path('/home/jovyan')

os.chdir(BASE_DIR)

for path_to_add in [str(BASE_DIR), str(BASE_DIR / 'src')]:
    if path_to_add not in sys.path:
        sys.path.insert(0, path_to_add)

DATA_DIR        = BASE_DIR / 'data' / 'raw'
SUBMISSIONS_DIR = BASE_DIR / 'submissions'
CHECKPOINTS_DIR = BASE_DIR / 'models' / 'checkpoints'
SRC_PATH        = BASE_DIR / 'src' / 'nllb_pipeline.py'

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'Training set.csv'
VAL_PATH   = DATA_DIR / 'Validation set.csv'
TEST_PATH  = DATA_DIR / 'Test set.csv'

print(f'BASE_DIR : {BASE_DIR.resolve()}')
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, SRC_PATH]:
    status = 'OK' if p.exists() else 'MISSING'
    rel_p = p.relative_to(BASE_DIR) if p.is_relative_to(BASE_DIR) else p
    print(f'  [{status}] {rel_p}')


In [ ]:
# 4. Load Data & Preview Language Subsets
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f'Train shape : {train_df.shape}')
print(f'Val shape   : {val_df.shape}')
print(f'Test shape  : {test_df.shape}')

print('\nSubset distribution in train data:')
display(train_df['subset'].value_counts())


In [ ]:
# 5. Interactive Test of Hybrid Dense + Lexical RAG Retriever & Threshold Grid
print('Initializing Hybrid RAG Retriever (TF-IDF + Dense Multilingual Embeddings)...')
hybrid_retriever = HybridRetriever(train_df, enable_dense=True)

val_retr_preds, val_retr_sims = [], []
for _, row in val_df.iterrows():
    ans, sim = hybrid_retriever.get_top1(row['input'], row['subset'], exclude_exact=True)
    val_retr_preds.append(ans)
    val_retr_sims.append(sim)

print(f'Retrieved {len(val_retr_preds)} reference answers for validation questions.')


In [ ]:
# 6. Fast Dry-Run Verification (Testing Pipeline End-to-End)
dry_run_cmd = [
    sys.executable, str(SRC_PATH),
    '--dry_run',
    '--skip_submission',
    '--model_name', 'facebook/nllb-200-distilled-600M',
]

print('Running fast dry-run verification with live log streaming:')
process = subprocess.Popen(
    dry_run_cmd, cwd=str(BASE_DIR),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
assert process.returncode == 0, f'Dry run failed with code {process.returncode}'


In [ ]:
# 8. Evaluate Best Checkpoint & Optionally Generate Submission File
# Evaluates validation metrics first. Set GENERATE_SUBMISSION = True when validation results are promising!
import json, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

CHECKPOINT_ROOT = CHECKPOINTS_DIR / 'nllb-health-qa-checkpoint'

def find_best_checkpoint(checkpoint_root: Path, fallback_model: str = 'facebook/nllb-200-distilled-600M'):
    if not checkpoint_root.exists():
        print(f"[NOTE] Checkpoint folder {checkpoint_root} not found. Using base model '{fallback_model}'.")
        return fallback_model
    state_files = sorted(checkpoint_root.glob('checkpoint-*/trainer_state.json'),
                          key=lambda p: p.stat().st_mtime, reverse=True)
    if not state_files:
        if (checkpoint_root / 'config.json').exists():
            return checkpoint_root
        print(f"[NOTE] No fine-tuned checkpoints found under {checkpoint_root}. Using base model '{fallback_model}'.")
        return fallback_model
    try:
        state = json.loads(state_files[0].read_text(encoding='utf-8'))
        best = state.get('best_model_checkpoint')
        if best and Path(best).exists():
            return Path(best)
        return state_files[0].parent
    except Exception as e:
        return state_files[0].parent

best_ckpt = find_best_checkpoint(CHECKPOINT_ROOT)
print('Model checkpoint loaded:', best_ckpt)

GENERATE_SUBMISSION = False
sub_path = SUBMISSIONS_DIR / 'submission_nllb_hybrid.csv'

if GENERATE_SUBMISSION:
    print('Generating predictions for Test Set...')
    sub_cmd = [
        sys.executable, str(SRC_PATH),
        '--model_name', str(best_ckpt),
        '--use_rag',
        '--use_dense_rag',
        '--train_path', str(TRAIN_PATH),
        '--val_path', str(VAL_PATH),
        '--test_path', str(TEST_PATH),
        '--submission_path', str(sub_path),
    ]
    process = subprocess.Popen(sub_cmd, cwd=str(BASE_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    if sub_path.exists():
        sub_df = pd.read_csv(sub_path)
        print(f'\n✅ Submission shape: {sub_df.shape}')
        display(sub_df.head(10))
else:
    print('GENERATE_SUBMISSION is False — Review validation metrics above. Set GENERATE_SUBMISSION = True to create submission CSV.')


In [ ]:
# 8. Generate Submission File (ONLY Run When Validation Results Are Promising)
# Set GENERATE_SUBMISSION = True when you confirm the validation ROUGE scores above are promising
GENERATE_SUBMISSION = False

sub_path = SUBMISSIONS_DIR / 'submission_nllb_hybrid.csv'

if GENERATE_SUBMISSION:
    print('Generating predictions for Test Set...')
    sub_cmd = [
        sys.executable, str(SRC_PATH),
        '--model_name', 'facebook/nllb-200-distilled-600M',
        '--use_rag',
        '--use_dense_rag',
        '--train_path', str(TRAIN_PATH),
        '--val_path', str(VAL_PATH),
        '--test_path', str(TEST_PATH),
        '--submission_path', str(sub_path),
    ]
    process = subprocess.Popen(sub_cmd, cwd=str(BASE_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    
    if sub_path.exists():
        sub_df = pd.read_csv(sub_path)
        print(f'\n✅ Submission shape: {sub_df.shape}')
        display(sub_df.head(10))
else:
    print('GENERATE_SUBMISSION is False — Review validation metrics above. Set GENERATE_SUBMISSION = True to create submission CSV.')


In [ ]:
# 9. Optional File Download (Google Colab)
try:
    from google.colab import files
    if GENERATE_SUBMISSION and sub_path.exists():
        files.download(str(sub_path))
except ImportError:
    pass
